# 🌾 Track 3 — Mandi-to-Market Supply Chain Optimizer
## 01 — Data Profiling


In [3]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT = Path("..")
RAW = ROOT / "data" / "raw"
OUT = ROOT / "outputs" / "profiling"
OUT.mkdir(parents=True, exist_ok=True)

arrivals = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_arrivals.csv")
master = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_master.csv")
transport = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_transport_logistics.csv")
master
with open(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_price_and_msp.json", encoding="utf-8") as f:
    prices = pd.DataFrame(json.load(f))

weather = pd.read_excel(RAW / "C:\\Users\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_weather_sensors.xlsx")

datasets = {
    "arrivals": arrivals,
    "prices": prices,
    "weather": weather,
    "transport": transport,
    "mandi_master": master,
}

print("Loaded datasets:")
for name, df in datasets.items():
    print(f"{name:15s} {df.shape[0]:>8,} rows × {df.shape[1]:>2} columns")


Loaded datasets:
arrivals          25,750 rows ×  8 columns
prices            12,000 rows ×  9 columns
weather           15,000 rows ×  7 columns
transport         10,400 rows × 10 columns
mandi_master          60 rows ×  6 columns


## 1. Dataset structure

In [4]:
for name, df in datasets.items():
    print(f"\n### {name}")
    display(pd.DataFrame({
        "column": df.columns,
        "dtype": [str(x) for x in df.dtypes],
        "missing_n": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(2),
        "unique_n": df.nunique(dropna=True).values,
    }))



### arrivals


,column,dtype,missing_n,missing_pct,unique_n
0,arrival_id,str,484,1.88,24524
1,date,str,0,0.00,1512
2,mandi_id,str,0,0.00,342
3,crop_name,str,0,0.00,36
4,variety,str,3735,14.50,6
5,arrival_quantity,str,0,0.00,23609
6,unit,str,5143,19.97,14
7,farmer_count,float64,3899,15.14,146



### prices


,column,dtype,missing_n,missing_pct,unique_n
0,record_id,str,0,0.00,12000
1,date,str,0,0.00,1511
2,mandi_id,str,1235,10.29,342
3,district,str,773,6.44,14
4,crop_name,str,0,0.00,36
5,min_price,object,0,0.00,10163
6,max_price,object,0,0.00,10403
7,modal_price,object,0,0.00,10230
8,msp,object,0,0.00,37



### weather


,column,dtype,missing_n,missing_pct,unique_n
0,sensor_id,str,0,0.00,51
1,timestamp,str,1555,10.37,13369
2,temperature,object,0,0.00,753
3,temp_unit,str,2263,15.09,8
4,rainfall,float64,791,5.27,1165
5,rain_unit,str,791,5.27,6
6,humidity_percent,float64,1500,10.00,66



### transport


,column,dtype,missing_n,missing_pct,unique_n
0,trip_id,str,0,0.00,10000
1,mandi_id,str,0,0.00,342
2,destination_warehouse,str,0,0.00,6
3,departure_time,str,0,0.00,9315
4,arrival_time,str,1053,10.12,8353
5,transit_hours,str,518,4.98,617
6,distance,str,0,0.00,7939
7,distance_unit,str,1032,9.92,2
8,vehicle_no,str,1622,15.60,8438
9,driver_id,str,1551,14.91,899



### mandi_master


,column,dtype,missing_n,missing_pct,unique_n
0,mandi_id,str,0,0.00,57
1,mandi_name,str,0,0.00,57
2,district,str,4,6.67,27
3,state,str,4,6.67,3
4,mandi_type,str,11,18.33,5
5,total_area_acres,float64,6,10.00,34


## 2. Exact duplicate records

In [5]:
duplicate_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "duplicate_pct": round(df.duplicated().mean() * 100, 2),
    }
    for name, df in datasets.items()
])

display(duplicate_summary)
duplicate_summary.to_csv(OUT / "duplicate_summary.csv", index=False)


,dataset,rows,exact_duplicate_rows,duplicate_pct
0,arrivals,25750,750,2.91
1,prices,12000,0,0.00
2,weather,15000,0,0.00
3,transport,10400,400,3.85
4,mandi_master,60,3,5.00


## 3. High-risk messy fields
Inspect the fields called out by the organizers: crop names, mandi IDs, quantities/units, prices, timestamps/timezones, weather units, transit times, distances and vehicle numbers.

In [6]:
checks = {
    "arrivals_mandi_id_variants": arrivals["mandi_id"].value_counts().head(50),
    "arrivals_crop_variants": arrivals["crop_name"].value_counts(),
    "arrivals_units": arrivals["unit"].value_counts(dropna=False),
    "transport_distance_units": transport["distance_unit"].value_counts(dropna=False),
    "weather_temp_units": weather["temp_unit"].value_counts(dropna=False),
    "weather_rain_units": weather["rain_unit"].value_counts(dropna=False),
    "transport_warehouses": transport["destination_warehouse"].value_counts(),
    "master_mandi_types": master["mandi_type"].value_counts(dropna=False),
}

for title, result in checks.items():
    print(f"\n### {title}")
    display(result)



### arrivals_mandi_id_variants


mandi_id
MANDI002    233
MANDI022    226
MANDI051    225
MANDI048    223
MANDI026    220
MANDI040    220
MANDI056    220
MANDI037    219
MANDI020    218
MANDI019    216
MANDI007    215
MANDI043    215
MANDI031    215
MANDI032    214
MANDI041    213
MANDI033    213
MANDI001    213
MANDI011    212
MANDI013    212
MANDI021    212
MANDI057    212
MANDI036    212
MANDI039    211
MANDI052    211
MANDI003    210
MANDI053    210
MANDI009    209
MANDI042    208
MANDI016    207
MANDI035    207
MANDI047    206
MANDI027    206
MANDI045    205
MANDI050    204
MANDI049    204
MANDI029    204
MANDI018    204
MANDI010    202
MANDI055    201
MANDI012    201
MANDI054    199
MANDI006    198
MANDI004    198
MANDI034    197
MANDI008    196
MANDI017    193
MANDI030    193
MANDI014    192
MANDI025    191
MANDI023    189
Name: count, dtype: int64


### arrivals_crop_variants


crop_name
Sugarcane    918
Cotton       913
Mustard      909
गन्ना        896
Sarson       887
Sarso        874
कपास         863
sugarcane    851
सरसों        843
Narma        837
Ganna        831
mustard      816
cotton       812
Kapas        790
Ganne        782
Corn         749
मक्का        729
Makka        727
Maize        722
corn         705
wheat        668
गेहूं        661
Makki        645
Wheat        644
Gehun        638
Kanak        624
WHEAT        609
GEHUN        587
Chawal       575
paddy        551
Paddy        551
चावल         551
Basmati      536
Rice         499
धान          485
Dhaan        472
Name: count, dtype: int64


### arrivals_units


...

rain_unit
mm             4459
MM             3038
millimeters    2916
inches         1285
in             1270
inch           1241
NaN             791
Name: count, dtype: int64


### transport_warehouses


destination_warehouse
WH-North           1782
WH-South           1777
WH-West            1746
WH-Central         1725
Export-Terminal    1702
WH-East            1668
Name: count, dtype: int64


### master_mandi_types


mandi_type
APMC       15
NaN        11
Private    10
Direct     10
PRIVATE     8
apmc        6
Name: count, dtype: int64

## 4. Raw-to-cleaning evidence summary

In [7]:
quality = pd.DataFrame([
    {
        "dataset": name,
        "raw_rows": len(df),
        "columns": len(df.columns),
        "exact_duplicates": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_cell_pct": round(df.isna().mean().mean() * 100, 2),
    }
    for name, df in datasets.items()
])

display(quality)
quality.to_csv(OUT / "raw_quality_summary.csv", index=False)

print("\nProfiling complete. Next step: define and document cleaning rules before changing any raw data.")


,dataset,raw_rows,columns,exact_duplicates,missing_cells,missing_cell_pct
0,arrivals,25750,8,750,13261,6.44
1,prices,12000,9,0,2008,1.86
2,weather,15000,7,0,6900,6.57
3,transport,10400,10,400,5776,5.55
4,mandi_master,60,6,3,25,6.94



Profiling complete. Next step: define and document cleaning rules before changing any raw data.
